# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

**Lane:** Lane 2 — *Refresh / Content Opportunity Scoring* (`docs/ml-intern-dataset-and-lane-guide.md` §8).

**The frame in one paragraph:** *For a content reviewer deciding which 50 pages to open first this week,
we will build a ranked action queue from trailing-90-day search and engagement signals, scoring each page
by how likely its visibility is to be declining. A wrong call costs editor hours on a page that did not need
work, or leaves a declining page with real demand unreviewed for another cycle. A plain rule is not enough
because the evidence is many weak, interacting signals and the output has to be an order, not a flag. We
will claim only observed, directional, decision-support results.*

The four sections answer the card in order, and each code cell shows the number the prose is built on.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.** The question I am taking into the capstone:

> *Which pages should a content reviewer open first this week — and what should they do with each one?*

Why this lane, in three honest reasons:

1. **The data already contains the decision's raw material.** One row per page with impressions, clicks,
   CTR, average position, freshness, and a trend label — enough to build and rank a queue today, before
   the warehouse is touched.
2. **The warehouse upgrades the target instead of changing the question.** Daily history lets the capstone
   replace the starter's rule-based label with a forward-looking outcome window (ML-03 records that split),
   so the 7 weeks buy measurement quality, not a different project.
3. **The output is a thing a human does on Monday.** A ranked queue with a suggested action and reason
   codes — not a dashboard without an owner, and not a model score nobody can disagree with.

It is also where my first two notebooks already pointed: notebook 01 found the CTR/type gap is a
launch-window effect, and notebook 02 showed the hand rule fires on only 17 of 30,000 pages — the real
problem is ordering, not flagging. I can confirm or change lanes until the end of Week 4; nothing below
depends on the warehouse, so the framing can be checked today.

In [1]:
import os
import sys

if "google.colab" in sys.modules:            # Colab: bring the repo in so the paths resolve
    if not os.path.isdir("flyrank-ml-internship"):
        !git clone --depth 1 https://github.com/Louai-Mohamed/flyrank-ml-internship.git
    os.chdir("flyrank-ml-internship")
else:                                        # local: walk up to the repo root
    while os.path.isdir("..") and not os.path.isdir("data/raw"):
        os.chdir("..")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
REVIEW_CAPACITY = 50          # pages one reviewer can work through in a week

weeks = len(df) / REVIEW_CAPACITY
print(f"inventory: {len(df):,} pages across {df['client_id'].nunique()} clients")
print(f"review capacity assumed: {REVIEW_CAPACITY} pages/week")
print(f"one full sweep of the inventory: {weeks:,.0f} weeks ({weeks / 52:.1f} years)")
print(f"-> each week has to choose {REVIEW_CAPACITY / len(df):.2%} of the inventory. That is triage,")
print("   which is why the lane is ranking / scoring, not classification.")

inventory: 30,000 pages across 32 clients
review capacity assumed: 50 pages/week
one full sweep of the inventory: 600 weeks (11.5 years)
-> each week has to choose 0.17% of the inventory. That is triage,
   which is why the lane is ranking / scoring, not classification.


## 2. The question: decision, action, cost of a wrong call

**The decision.** Each week a content reviewer has a fixed number of slots (I assume 50). Which pages do
they open first? Today that choice is made from memory, a spreadsheet sort, or whoever shouted loudest.

**Who acts, and what they do.** A content editor or SEO lead works down the queue and takes one of a small
set of actions per page: *refresh* (update facts, examples, internal links), *expand* (thin page with
demand), *fix CTR* (visible but under-clicked), *protect* (strong page with early decay signals), or
*monitor*. Each queue row carries a score, a suggested action and reason codes, so the reviewer can
inspect and overrule the recommendation.

**What a wrong call costs.** The errors are asymmetric:

- **A false positive** (a page that was not declining, or that nobody could improve) burns 2–4 editor
  hours — an assumption I will keep visible, not a measured cost.
- **A false negative** (a declining page with real demand never reviewed) leaves demand unserved for at
  least another cycle, and the queue's job is precisely to avoid that.

**What the queue is not.** It is not a promise that a refresh will recover a page. Proving that needs an
experiment this data cannot provide (lane guide §6, §7). The output allocates *review attention* — the
claim stops there.

In [2]:
down = df[df["trend_direction"] == "down"]
high_demand = down[down["impressions_90d"] >= 1000]
quarter_slots = REVIEW_CAPACITY * 13

print(f"declining pages: {len(down):,} ({len(down) / len(df):.1%} of the inventory)")
print(f"declining AND at least 1,000 impressions: {len(high_demand):,} "
      f"({len(high_demand) / len(down):.1%} of declining pages)")
print(f"impressions sitting on declining pages: "
      f"{down['impressions_90d'].sum() / df['impressions_90d'].sum():.1%} of the slice total\n")

print(f"one quarter of review slots: {quarter_slots:,} pages = "
      f"{quarter_slots / len(down):.1%} of the declining set")
print(f"-> even a perfect quarter touches only a fraction, so ORDER is the product.\n")

hours_low, hours_high = 2, 4      # explicit assumptions, not measurements
print("cost of a wrong call, assumptions stated:")
print(f"  a refresh takes ~{hours_low}-{hours_high} editor hours (assumption)")
print(f"  a wrong top-50 pick burns {hours_low}-{hours_high} hours of a {REVIEW_CAPACITY}-slot week "
      f"({hours_low / (REVIEW_CAPACITY * 8):.1%}-{hours_high / (REVIEW_CAPACITY * 8):.1%} of a 40-hour week)")
print(f"  a missed high-demand page leaves one of {len(high_demand):,} declining pages with real "
      "demand unreviewed")

declining pages: 16,262 (54.2% of the inventory)
declining AND at least 1,000 impressions: 8,031 (49.4% of declining pages)
impressions sitting on declining pages: 51.3% of the slice total

one quarter of review slots: 650 pages = 4.0% of the declining set
-> even a perfect quarter touches only a fraction, so ORDER is the product.

cost of a wrong call, assumptions stated:
  a refresh takes ~2-4 editor hours (assumption)
  a wrong top-50 pick burns 2-4 hours of a 50-slot week (0.5%-1.0% of a 40-hour week)
  a missed high-demand page leaves one of 8,031 declining pages with real demand unreviewed


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter slice that make this lane worth the next 7 weeks — all computed live below.

**Number 1 — the label is not rare, so accuracy is a trap.** 54.2% of pages are `down`. A model that
answers "declining" every time is right more than half the time, which is why the plan measures
Precision@50 against the base rate instead of celebrating accuracy.

**Number 2 — demand is concentrated.** The top 10% of pages by impressions hold **70.2%** of all
impressions (the top 1% alone: 24.9%). A queue that spends its slots on low-demand pages wastes the one
scarce resource in the project: human review time.

**Number 3 — the signal is real but heterogeneous.** Among visible pages (≥ 1,000 impressions, position
1–20), CTR runs from 0.05% at the 10th percentile to 0.72% at the 90th — a **14.4x spread** at comparable
position. The Week-1 discovery has the same shape: inside `page_1`, comparison articles earn 0.08% CTR
against 0.35% for keyword articles (4.3x), but the gap is concentrated in pages under ~180 days old.
No single cutoff encodes that — the lane needs context, which is the opening for a model.

In [3]:
base = df["trend_direction"].str.lower().eq("down").mean()
print(f"1. base rate: {base:.1%} of pages are labelled down "
      f"({int(base * len(df)):,} of {len(df):,})")

imp = df["impressions_90d"].sort_values(ascending=False)
top10 = imp.head(len(df) // 10).sum() / imp.sum()
top1 = imp.head(len(df) // 100).sum() / imp.sum()
print(f"2. concentration: top 10% hold {top10:.1%} of impressions | top 1% hold {top1:.1%}")

visible = df[(df["impressions_90d"] >= 1000) & (df["avg_position"] > 0)
             & (df["avg_position"] <= 20)]
p10, p50, p90 = visible["ctr"].quantile([0.10, 0.50, 0.90])
spread = p90 / max(p10, 0.01)
print(f"3. CTR spread among {len(visible):,} visible pages: "
      f"p10={p10:.2f}%  p50={p50:.2f}%  p90={p90:.2f}%  ({spread:.1f}x)\n")

p1 = df[df["position_tier"] == "page_1"]
print("   same shape inside page_1 (weighted CTR = clicks / impressions):")
for t in ["keyword article", "comparison article"]:
    s = p1[p1["content_type"] == t]
    print(f"     {t:<20} n={len(s):>6,}  CTR={s['clicks_90d'].sum() / s['impressions_90d'].sum():.2%}")

1. base rate: 54.2% of pages are labelled down (16,262 of 30,000)
2. concentration: top 10% hold 70.2% of impressions | top 1% hold 24.9%
3. CTR spread among 10,011 visible pages: p10=0.05%  p50=0.22%  p90=0.72%  (14.4x)

   same shape inside page_1 (weighted CTR = clicks / impressions):
     keyword article      n=10,501  CTR=0.35%
     comparison article   n=   435  CTR=0.08%


## 4. Careful words: what I can and can't claim

**What the work will be able to say:**

- *observed* — associations in this anonymized slice (e.g. the concentration and CTR spreads above);
- *measured on held-out clients* — ranking quality such as Precision@50 and ROC-AUC, always beside the
  base rate and a client-holdout split (ML-03 already did this once);
- *directional* — when a lift is smaller than the split-to-split spread;
- *decision-support* — the queue orders human review; a human makes the call.

**What it will never claim:**

- that refreshing a page **causes** recovery — no experiment, no treatment effect;
- anything about "the Google algorithm" — these are 32 anonymized clients, not a ranking system;
- that a result here generalizes to every client — 32 clients is a small, self-selected sample;
- that the starter label is a future outcome — it is a rule over the trailing window (99.98%
  reproducible from ordinary-looking columns, as ML-03 proves), which is why the capstone moves to a
  forward window on the warehouse.

**The data's blind spots, counted rather than hand-waved:**

In [4]:
limits = pd.DataFrame([
    {"blind spot": "no previous-window impressions",
     "rows": int((df["impressions_prev_30d"] == 0).sum()),
     "consequence": "trend is 'new' or 'flat', not a decline"},
    {"blind spot": "no position data (avg_position = 0)",
     "rows": int((df["avg_position"] == 0).sum()),
     "consequence": "position-based rules silently skip them"},
    {"blind spot": "word_count missing",
     "rows": int(df["word_count"].isna().sum()),
     "consequence": "fillna(0) would inject a content-type signal"},
])
for row in limits.itertuples(index=False):
    print(f"  {row[0]:<38}{row[1]:>6,}  ->  {row[2]}")

claims = pd.DataFrame([
    ["demand is concentrated", "top 10% hold 70.2% of impressions", "observed"],
    ["one cutoff is not enough", "14.4x CTR spread at comparable position", "observed"],
    ["a ranking beats a coin flip", "ML-03: AUC 0.647 ± 0.061 vs 0.500", "measured, held-out clients"],
    ["refreshing recovers a page", "no experiment in this data", "never claimed"],
], columns=["claim", "evidence", "allowed wording"])
print("\nclaim sheet:")
for row in claims.itertuples(index=False):
    print(f"  {row[0]:<30}{row[1]:<48}{row[2]}")

  no previous-window impressions         3,388  ->  trend is 'new' or 'flat', not a decline
  no position data (avg_position = 0)    1,205  ->  position-based rules silently skip them
  word_count missing                     7,699  ->  fillna(0) would inject a content-type signal

claim sheet:
  demand is concentrated        top 10% hold 70.2% of impressions               observed
  one cutoff is not enough      14.4x CTR spread at comparable position         observed
  a ranking beats a coin flip   ML-03: AUC 0.647 ± 0.061 vs 0.500               measured, held-out clients
  refreshing recovers a page    no experiment in this data                      never claimed


## Self-check

- [x] Every section above is filled — markdown thinking **and** the code that backs it
- [x] The notebook runs top to bottom with no errors (`Runtime → Run all`)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id` / `client_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card

**Lane status:** provisional, confirmable until the end of Week 4. The next steps that would change my
mind are data-contract findings (ML-04): if the warehouse forward window turns out to be too sparse for
most clients, I would narrow the lane to the starter slice's trailing-window queue rather than fake a
future label.